# Part B — Venus stereo and 3D reconstruction

Compare BM and SGBM on the **Middlebury Flow Venus** pair. SIFT correspondences feed RANSAC estimation of F and epipolar-line visualization. Signed disparity is evaluated against horizontal flow GT and reprojected with a pinhole camera using **30° horizontal FoV and baseline B=1**.

The notebook opens the saved full run. Set `rerun_pipeline = True` to recalculate the grid. Camera parameters and references are stored with the results.


In [ ]:
import copy
import html
import io
import json
import os
import sys
from pathlib import Path
from IPython.display import HTML, Image, display
from PIL import Image as PILImage

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src/part_b.py").is_file())
sys.path.insert(0, str(project_root / "src"))
import part_b

config = copy.deepcopy(part_b.DEFAULT_CONFIG)
rerun_pipeline = False
run_directory = None  # Có thể điền đường dẫn run lịch sử để kiểm tra riêng.

if rerun_pipeline:
    run_dir = part_b.run_experiments(project_root / part_b.DEFAULT_INPUT, project_root / part_b.DEFAULT_OUTPUT, config)
elif run_directory is not None:
    run_dir = Path(run_directory)
    if not run_dir.is_absolute():
        run_dir = project_root / run_dir
else:
    run_dir = part_b.find_matching_run(project_root / part_b.DEFAULT_INPUT, project_root / part_b.DEFAULT_OUTPUT, config)

validation = part_b.validate_run(run_dir)
manifest = json.loads((run_dir / "run.json").read_text())
config = manifest["config"]
camera = json.loads((run_dir / "camera.json").read_text())
figures = json.loads((run_dir / "figures.json").read_text())
rows = part_b.read_csv(run_dir / "metrics.csv")
geometry_rows = part_b.read_csv(run_dir / "geometry_metrics.csv")
print("Run:", run_dir)
print("Status:", manifest["status"])
print("Validation:", validation)
print("Camera:", camera["label"])
common_rows = part_b.read_csv(run_dir / "common_domain_metrics.csv")
print("Geometry protocol:", manifest["geometry_protocol"])
print("Warnings:", manifest["warnings"])


In [ ]:
def table(records, columns):
    header = "".join("<th>" + html.escape(column) + "</th>" for column in columns)
    body = []
    for record in records:
        values = []
        for column in columns:
            value = record.get(column)
            if value is None or value == "":
                value = "N/A"
            elif column.endswith(("_px", "_px2", "_pct", "_ms")):
                value = f"{float(value):.4f}"
            values.append("<td>" + html.escape(str(value)) + "</td>")
        body.append("<tr>" + "".join(values) + "</tr>")
    display(HTML("<div style='overflow-x:auto'><table><thead><tr>" + header + "</tr></thead><tbody>" + "".join(body) + "</tbody></table></div>"))


def show(name):
    relative = figures.get(name)
    if relative is None:
        print("Figure unavailable:", name)
    else:
        # Giới hạn preview; PNG đầy đủ vẫn nằm trong thư mục run.
        with PILImage.open(run_dir / relative) as picture:
            picture.thumbnail((1600, 1600))
            buffer = io.BytesIO()
            picture.save(buffer, format="PNG")
            display(Image(data=buffer.getvalue(), width=min(1150, picture.width)))


table([{"component": key, "value": value} for key, value in manifest["environment"].items()], ["component", "value"])
table([{"filename": name, **record} for name, record in manifest["input_metadata"]["files"].items()], ["filename", "role_in_source", "width", "height", "sha256"])
show("input_and_ground_truth")

## Coordinate and mask conventions

Inputs stay at 420×380 without an additional warp. `frame10` is left/reference and `frame11` is right/target. Flow goes from reference to target, whereas stereo disparity is `x_left - x_right`, so **`d_gt = -flow_u`**. The supplied flow has zero vertical component; valid disparity can be negative or zero. Native matcher output is fixed point and is divided by 16.

The GT evaluation domain contains finite/non-unknown flow whose target coordinate stays inside the right image. It is **not an occlusion ground-truth mask**. Raw validity uses each matcher's own invalid sentinel. LR consistency tests `abs(d_left + d_right) <= 1 px` at the rounded corresponding location.

Raw and LR-valid ratios use all image pixels as denominator. RMSE and bad-pixel rate use the prediction-valid intersection with the fixed GT domain; bad pixels have absolute error **greater than 1 px**. Coverage is shown alongside these conditional scores. `bad_or_invalid_pct` counts rejected predictions as failures over the fixed GT domain.

## Correspondence, RANSAC and epipolar geometry

SIFT with a 5,000 feature target, BF L2 KNN matching and Lowe ratio 0.75. Fundamental matrices are estimated with OpenCV's **USAC_MAGSAC RANSAC preset**, including native sigma-consensus local optimization. This preset has its own fixed native random state **0**, independently of the project seed **42** used elsewhere. Confidence is 0.999, maximum iterations 5,000, and native robust-estimation thresholds are 0.5/1/2 px. The 1 px case is the fixed reference. The estimator receives only feature correspondences; GT is used afterward for evaluation.

The epipolar score averages the two point-to-line distances. `mean_sampson_px` averages the square root of Sampson's squared approximation; the squared version is stored separately in px². All-match and RANSAC-inlier statistics are separate. These scores describe the feature correspondences; disparity accuracy is reported by RMSE and bad-pixel rate.

An independent check uses the entire GT-valid/in-bounds domain and measures line directions on both views. A p95 absolute angle above **2°** raises a warning because the input is rectified. The project uses a 2° p95 limit for this rectified pair. The estimated F and both angle distributions are saved. The dense branch uses the existing rectified images directly.


In [ ]:
table(geometry_rows, ["threshold_px", "status", "detected_left", "detected_right", "initial_matches", "ratio_pass_matches", "inliers", "inlier_ratio_pct", "inlier_mean_epipolar_px", "inlier_mean_sampson_px", "gt_domain_mean_epipolar_px", "gt_domain_p95_epipolar_px", "left_angle_p95_deg", "right_angle_p95_deg", "geometry_quality", "ransac_ms", "error"])
print("SIFT + KNN/ratio time (single sample, ms):", manifest.get("feature_matching_ms", "N/A"))
show("initial_matches")
show("inlier_matches")
show("geometry_validation")
for row in geometry_rows:
    if row["status"] == "ok":
        show(row["geometry_id"] + "_epipolar")


## Dense comparison and timing

The full grid is BM/SGBM × `numDisparities` 32/64 × `blockSize` 5/9/15, with `minDisparity=-16`. Both directions use the same settings except the mirrored search range. Uniqueness ratio is 10 and prefilter cap is 31; BM texture threshold is 10. SGBM uses 3-way aggregation and grayscale penalties `P1=8b²`, `P2=32b²`. Speckle filtering is off. The internal consistency threshold is set above the possible range; external LR consistency is applied separately.

One OpenCV thread and OpenCL disabled. Full runs perform three warmups and twenty measured repetitions. `left_matching_ms` measures only left native matching. `stereo_pipeline_ms` includes construction, both native passes, disparity conversion and consistency masks, excluding disk I/O, GT metrics, figures and 3D. Geometry times are single samples and are not added to dense timings. All individual samples are saved in `timings.csv`.
The CSV and tables include the dense pipeline minimum, maximum, population standard deviation and coefficient of variation (CV). Whiskers show observed min/max, not confidence intervals. Native disparity equality across repetitions is recorded separately from timing variation. A second table compares each BM/SGBM pair on their **identical common LR-valid intersection** with the fixed GT domain. Empty or failed comparisons retain an explicit N/A reason.


In [ ]:
table(rows, ["case_id", "status", "raw_valid_ratio_pct", "consistent_valid_ratio_pct", "raw_rmse_px", "rmse_px", "bad_pixel_pct", "coverage_pct", "bad_or_invalid_pct", "evaluated_pixels", "left_matching_ms", "stereo_pipeline_ms", "stereo_pipeline_min_ms", "stereo_pipeline_max_ms", "stereo_pipeline_std_ms", "stereo_pipeline_cv_pct", "timing_repetitions", "repeat_deterministic", "error"])
table(common_rows, ["case_id", "status", "common_pixels", "common_coverage_pct", "common_rmse_px", "common_bad_pixel_pct", "reason"])
show("metric_comparison")
show("bm_disparity_grid")
show("sgbm_disparity_grid")
print(figures["display_convention"])
for method in ["bm", "sgbm"]:
    show(method + "_n32_b9_diagnostics")


## Camera model and point clouds

The camera model uses the **30° viewing-angle choice of Basha, Moses and Kiryati (2013), §4.1**. This project applies it as horizontal FoV on the 434×383 source frame and selects baseline **B=1**, square pixels, zero skew, parallel camera axes and centered source principal points. Distances are expressed in **normalized baseline units**.

| Parameter | Value |
|---|---|
| Focal length | `434 / (2 tan(15°)) = 809.855025 px` |
| Left principal point | `(206.5, 191)` |
| Right principal point | `(216.5, 191)` |
| Left/right crop origin | `(10, 0)` / `(0, 0)` |
| Principal-point offset | `doffs = 10 px` |
| Coordinate frame | Left camera; X right, Y down, Z forward |

For cropped disparity `d`, `Z=f/(d+10)`, `X=(x-206.5)/(d+10)` and `Y=(y-191)/(d+10)`. Disparity-error metrics use `d_gt=-flow_u`; the crop correction is applied during reprojection.

The BM and SGBM point clouds both use the fixed configuration **ndisp=32, block=9**. Reprojection retains LR-consistent pixels with positive denominator, finite XYZ and positive depth.

`cloud.ply` contains the raw retained points. `cloud_filtered.ply` applies OpenCV `filterSpeckles` with connected components of at most **100 pixels** and neighboring disparity differences of at most **1 px**. Filtering operates on predicted disparity. Both point sets, masks, depth maps and removal counts are saved.

Figures show raw full extent, raw detail and filtered clouds. Detail bounds are shared across the filtered predictions. Raw and filtered depth figures use one shared color scale, with saturation counts in their titles.


In [ ]:
display(HTML("<pre>" + html.escape(json.dumps(camera, indent=2, ensure_ascii=False)) + "</pre>"))
print("Shared depth display limits:", figures.get("depth_display_limits"))
print(figures.get("cloud_detail_bounds_source", "N/A"))
for case_id, cloud in manifest["point_clouds"].items():
    print(case_id, cloud)
    if cloud["status"] == "ok":
        for suffix in ["_depth", "_depth_filtered", "_point_cloud", "_point_cloud_detail", "_point_cloud_filtered"]:
            show(case_id + suffix)
        for filename in ["cloud.ply", "cloud_filtered.ply"]:
            path = run_dir / "cases" / case_id / filename
            link = os.path.relpath(path, project_root / "notebooks")
            label = case_id + " / " + filename + " (assumed camera; normalized units)"
            display(HTML('<a href="' + html.escape(link, quote=True) + '">Download ' + html.escape(label) + '</a>'))


## Saved results and references

`run.json` records inputs, configuration, timing scope, environment and artifact hashes. `geometry/` holds feature arrays, correspondence coordinates, F and inlier masks. Each `cases/` directory contains native and floating-point disparities, validity masks, matcher settings and point clouds. `camera.json`, `references.json`, CSV tables, figures and the measured source snapshot accompany the run.


In [ ]:
references = json.loads((run_dir / "references.json").read_text())
for key, reference in references.items():
    print(key)
    print(json.dumps(reference, ensure_ascii=False, indent=2))
print("Final validation:", part_b.validate_run(run_dir))